# 🔀 Notebook 3: Sharding and Partitioning

When one server can't handle the write load, distribute it across multiple servers. The key is choosing how to split the data.

## Learning Objectives

By the end of this notebook, you'll understand:
- Horizontal vs vertical partitioning
- Choosing effective partition keys
- Avoiding hot spots
- Consistent hashing basics

In [1]:
import hashlib
import random
from collections import defaultdict
from typing import List, Dict

print("✅ Ready to learn about sharding!")

✅ Ready to learn about sharding!


## 🔀 Horizontal Sharding

In [2]:
print("🔀 Horizontal Sharding")
print("=" * 60)
print("""
Split ROWS across multiple databases based on a key.

BEFORE (Single DB):
─────────────────────────────────────────────────────────────
┌─────────────────────────────────────────┐
│            All Posts                    │
│  user_id=1, user_id=2, ... user_id=N   │
│         (Bottleneck!)                   │
└─────────────────────────────────────────┘

AFTER (Sharded by user_id):
─────────────────────────────────────────────────────────────
┌─────────────┐  ┌─────────────┐  ┌─────────────┐
│   Shard 0   │  │   Shard 1   │  │   Shard 2   │
│ user_id % 3 │  │ user_id % 3 │  │ user_id % 3 │
│    = 0      │  │    = 1      │  │    = 2      │
└─────────────┘  └─────────────┘  └─────────────┘

• Each shard handles 1/3 of the writes
• Linear scaling: 3 shards = 3x capacity
""")

🔀 Horizontal Sharding

Split ROWS across multiple databases based on a key.

BEFORE (Single DB):
─────────────────────────────────────────────────────────────
┌─────────────────────────────────────────┐
│            All Posts                    │
│  user_id=1, user_id=2, ... user_id=N   │
│         (Bottleneck!)                   │
└─────────────────────────────────────────┘

AFTER (Sharded by user_id):
─────────────────────────────────────────────────────────────
┌─────────────┐  ┌─────────────┐  ┌─────────────┐
│   Shard 0   │  │   Shard 1   │  │   Shard 2   │
│ user_id % 3 │  │ user_id % 3 │  │ user_id % 3 │
│    = 0      │  │    = 1      │  │    = 2      │
└─────────────┘  └─────────────┘  └─────────────┘

• Each shard handles 1/3 of the writes
• Linear scaling: 3 shards = 3x capacity



In [3]:
class SimpleShardedDB:
    def __init__(self, num_shards: int):
        self.num_shards = num_shards
        self.shards = {i: [] for i in range(num_shards)}
        self.write_counts = {i: 0 for i in range(num_shards)}
    
    def get_shard(self, key: int) -> int:
        return key % self.num_shards
    
    def write(self, user_id: int, data: dict):
        shard_id = self.get_shard(user_id)
        self.shards[shard_id].append({"user_id": user_id, **data})
        self.write_counts[shard_id] += 1
    
    def get_distribution(self) -> dict:
        total = sum(self.write_counts.values())
        return {
            shard_id: {
                "count": count,
                "percentage": (count / total * 100) if total > 0 else 0
            }
            for shard_id, count in self.write_counts.items()
        }

print("🔬 Simulating Writes with Uniform User IDs")
print("=" * 60)

db = SimpleShardedDB(num_shards=4)

for user_id in range(1, 10001):
    db.write(user_id, {"action": "post"})

print("\n📊 Write Distribution (uniform user IDs):")
for shard_id, stats in db.get_distribution().items():
    bar = "█" * int(stats["percentage"] / 2)
    print(f"   Shard {shard_id}: {stats['count']:>5} writes ({stats['percentage']:.1f}%) {bar}")

print("\n✅ Even distribution when keys are uniform!")

🔬 Simulating Writes with Uniform User IDs

📊 Write Distribution (uniform user IDs):
   Shard 0:  2500 writes (25.0%) ████████████
   Shard 1:  2500 writes (25.0%) ████████████
   Shard 2:  2500 writes (25.0%) ████████████
   Shard 3:  2500 writes (25.0%) ████████████

✅ Even distribution when keys are uniform!


## ⚠️ The Hot Spot Problem

In [4]:
print("⚠️ Bad Partition Key: Country")
print("=" * 60)

class CountryShardedDB:
    def __init__(self):
        self.shards = defaultdict(list)
        self.write_counts = defaultdict(int)
    
    def write(self, country: str, data: dict):
        self.shards[country].append(data)
        self.write_counts[country] += 1

db = CountryShardedDB()

countries = {
    "USA": 330,
    "China": 1400,
    "India": 1380,
    "Brazil": 210,
    "Russia": 140,
    "Japan": 125,
    "Germany": 83,
    "UK": 67,
    "New Zealand": 5,
    "Iceland": 0.3
}

for country, population in countries.items():
    writes = int(population * 10)
    for _ in range(writes):
        db.write(country, {"action": "post"})

total = sum(db.write_counts.values())
print("\n📊 Write Distribution by Country:")
for country in sorted(db.write_counts.keys(), key=lambda x: db.write_counts[x], reverse=True):
    count = db.write_counts[country]
    pct = count / total * 100
    bar = "█" * int(pct / 2)
    print(f"   {country:12}: {count:>6} writes ({pct:>5.1f}%) {bar}")

print("\n❌ China and India get 75% of writes!")
print("   This creates HOT SPOTS - some shards overloaded!")

⚠️ Bad Partition Key: Country

📊 Write Distribution by Country:
   China       :  14000 writes ( 37.4%) ██████████████████
   India       :  13800 writes ( 36.9%) ██████████████████
   USA         :   3300 writes (  8.8%) ████
   Brazil      :   2100 writes (  5.6%) ██
   Russia      :   1400 writes (  3.7%) █
   Japan       :   1250 writes (  3.3%) █
   Germany     :    830 writes (  2.2%) █
   UK          :    670 writes (  1.8%) 
   New Zealand :     50 writes (  0.1%) 
   Iceland     :      3 writes (  0.0%) 

❌ China and India get 75% of writes!
   This creates HOT SPOTS - some shards overloaded!


## 🎯 Choosing Good Partition Keys

In [5]:
print("🎯 Characteristics of Good Partition Keys")
print("=" * 60)
print("""
GOOD PARTITION KEYS:
─────────────────────────────────────────────────────────────
✅ High cardinality (many unique values)
✅ Uniform distribution of writes
✅ Matches your access patterns
✅ Doesn't change often

Examples:
• user_id - Good for user-centric data
• order_id - Good for e-commerce
• device_id - Good for IoT

─────────────────────────────────────────────────────────────

BAD PARTITION KEYS:
─────────────────────────────────────────────────────────────
❌ Low cardinality (few values)
❌ Skewed distribution
❌ Time-based (all writes go to "current" shard)
❌ Frequently changing

Examples:
• country - Skewed (China >> Iceland)
• status - Low cardinality (pending/active/done)
• timestamp - All "now" writes go to same shard
• celebrity_id - One key gets millions of writes
""")

🎯 Characteristics of Good Partition Keys

GOOD PARTITION KEYS:
─────────────────────────────────────────────────────────────
✅ High cardinality (many unique values)
✅ Uniform distribution of writes
✅ Matches your access patterns
✅ Doesn't change often

Examples:
• user_id - Good for user-centric data
• order_id - Good for e-commerce
• device_id - Good for IoT

─────────────────────────────────────────────────────────────

BAD PARTITION KEYS:
─────────────────────────────────────────────────────────────
❌ Low cardinality (few values)
❌ Skewed distribution
❌ Time-based (all writes go to "current" shard)
❌ Frequently changing

Examples:
• country - Skewed (China >> Iceland)
• status - Low cardinality (pending/active/done)
• timestamp - All "now" writes go to same shard
• celebrity_id - One key gets millions of writes



In [6]:
print("🎲 Hash-Based Sharding")
print("=" * 60)

def hash_shard(key: str, num_shards: int) -> int:
    hash_value = int(hashlib.md5(str(key).encode()).hexdigest(), 16)
    return hash_value % num_shards

db = SimpleShardedDB(num_shards=4)
db.get_shard = lambda user_id: hash_shard(user_id, 4)

for country, population in countries.items():
    writes = int(population * 10)
    for i in range(writes):
        user_id = f"{country}_{i}"
        shard = hash_shard(user_id, 4)
        db.write_counts[shard] += 1

total = sum(db.write_counts.values())
print("\n📊 Distribution with Hash-Based Sharding:")
for shard_id in sorted(db.write_counts.keys()):
    count = db.write_counts[shard_id]
    pct = count / total * 100
    bar = "█" * int(pct / 2)
    print(f"   Shard {shard_id}: {count:>6} writes ({pct:>5.1f}%) {bar}")

print("\n✅ Hash spreads writes evenly regardless of input distribution!")

🎲 Hash-Based Sharding

📊 Distribution with Hash-Based Sharding:
   Shard 0:   9385 writes ( 25.1%) ████████████
   Shard 1:   9223 writes ( 24.7%) ████████████
   Shard 2:   9339 writes ( 25.0%) ████████████
   Shard 3:   9456 writes ( 25.3%) ████████████

✅ Hash spreads writes evenly regardless of input distribution!


## 🧭 Modulo Sharding Breaks When You Add a Node

`user_id % N` is simple, but **changing `N` remaps almost every key**.
If you grow from 4 → 5 shards, roughly 4 out of every 5 keys move. In
practice that means a multi-hour rebalancing storm.

```
N=4:  key 42 -> shard 42 % 4 = 2
N=5:  key 42 -> shard 42 % 5 = 2   ✅ same shard (lucky!)
N=4:  key 17 -> shard 17 % 4 = 1
N=5:  key 17 -> shard 17 % 5 = 2   ❌ moved
```

**Consistent hashing** is the classic fix: when you add or remove a node,
only `K/N` keys move instead of almost all of them.


## ⭕ Consistent Hashing (The Intuition)

Imagine all possible hashes laid out on a **circle** (0 -> 2^32 -> 0 again).
Every shard gets placed at several positions on that circle (these are
called **virtual nodes**, or *vnodes*).

To find which shard owns a key:

1. Hash the key -> a point on the circle.
2. Walk **clockwise** until you hit a shard's vnode.
3. That shard owns the key.

Adding a new shard only "steals" the slice of circle next to its vnodes —
most keys stay where they are.

This is how Cassandra, DynamoDB, and many CDN load balancers route data.


In [ ]:
from bisect import bisect

class ConsistentHashRing:
    """Minimal consistent-hash ring with virtual nodes.

    Each real shard owns vnodes_per_shard points on the ring. More vnodes
    -> smoother load distribution. Typical values: 100-500 per shard.
    """

    def __init__(self, shards, vnodes_per_shard: int = 100):
        self.vnodes_per_shard = vnodes_per_shard
        self.ring = {}
        self.sorted_points = []
        for s in shards:
            self.add_shard(s)

    def _hash(self, s: str) -> int:
        return int(hashlib.md5(s.encode()).hexdigest(), 16)

    def add_shard(self, shard_id):
        for v in range(self.vnodes_per_shard):
            point = self._hash(f"{shard_id}#{v}")
            self.ring[point] = shard_id
        self.sorted_points = sorted(self.ring.keys())

    def remove_shard(self, shard_id):
        for v in range(self.vnodes_per_shard):
            point = self._hash(f"{shard_id}#{v}")
            self.ring.pop(point, None)
        self.sorted_points = sorted(self.ring.keys())

    def get_shard(self, key) -> str:
        point = self._hash(str(key))
        i = bisect(self.sorted_points, point) % len(self.sorted_points)
        return self.ring[self.sorted_points[i]]


print("⭕ Consistent Hashing: adding a shard with minimal movement")
print("=" * 60)

ring4 = ConsistentHashRing(["A", "B", "C", "D"], vnodes_per_shard=200)
ring5 = ConsistentHashRing(["A", "B", "C", "D", "E"], vnodes_per_shard=200)

keys = [f"user_{i}" for i in range(10_000)]

dist4 = defaultdict(int)
for k in keys:
    dist4[ring4.get_shard(k)] += 1

print("\n📊 With 4 shards (A-D):")
for s, n in sorted(dist4.items()):
    bar = "#" * (n // 100)
    print(f"   Shard {s}: {n:>5} ({n/len(keys)*100:.1f}%) {bar}")

moved = sum(1 for k in keys if ring4.get_shard(k) != ring5.get_shard(k))
modulo_moved = sum(1 for i in range(len(keys)) if (i % 4) != (i % 5))

print(f"\n🔁 Keys that move when going 4 -> 5 shards:")
print(f"   Consistent hash: {moved:>5} / {len(keys)} ({moved/len(keys)*100:.1f}%)")
print(f"   Plain modulo:    {modulo_moved:>5} / {len(keys)} ({modulo_moved/len(keys)*100:.1f}%)")
print("\n✅ Consistent hashing ~ K/N keys move; modulo moves almost everything.")


## 📐 Vertical Partitioning

In [7]:
print("📐 Vertical Partitioning")
print("=" * 60)
print("""
Split COLUMNS into different tables/databases based on access patterns.

BEFORE (Monolithic):
─────────────────────────────────────────────────────────────
┌─────────────────────────────────────────────────────────┐
│                        posts                             │
├──────┬─────────┬───────────┬────────────┬───────────────┤
│  id  │ content │ like_cnt  │ view_cnt   │ share_cnt     │
│      │ (write  │ (frequent │ (very      │ (occasional   │
│      │  once)  │  updates) │  frequent) │  updates)     │
└──────┴─────────┴───────────┴────────────┴───────────────┘

Problem: Like/view updates cause locks on content reads!

AFTER (Vertically Partitioned):
─────────────────────────────────────────────────────────────
┌─────────────────┐    ┌──────────────────────────────────┐
│  post_content   │    │         post_metrics             │
├──────┬──────────┤    ├──────┬─────────┬────────┬────────┤
│  id  │ content  │    │  id  │like_cnt │view_cnt│share_ct│
│      │          │    │      │         │        │        │
│ (write once,    │    │ (high-frequency counter updates) │
│  read often)    │    │                                  │
└─────────────────┘    └──────────────────────────────────┘

• Different write patterns = different optimizations
• Content: Optimized for reads (more indexes)
• Metrics: Optimized for writes (minimal indexes)
""")

📐 Vertical Partitioning

Split COLUMNS into different tables/databases based on access patterns.

BEFORE (Monolithic):
─────────────────────────────────────────────────────────────
┌─────────────────────────────────────────────────────────┐
│                        posts                             │
├──────┬─────────┬───────────┬────────────┬───────────────┤
│  id  │ content │ like_cnt  │ view_cnt   │ share_cnt     │
│      │ (write  │ (frequent │ (very      │ (occasional   │
│      │  once)  │  updates) │  frequent) │  updates)     │
└──────┴─────────┴───────────┴────────────┴───────────────┘

Problem: Like/view updates cause locks on content reads!

AFTER (Vertically Partitioned):
─────────────────────────────────────────────────────────────
┌─────────────────┐    ┌──────────────────────────────────┐
│  post_content   │    │         post_metrics             │
├──────┬──────────┤    ├──────┬─────────┬────────┬────────┤
│  id  │ content  │    │  id  │like_cnt │view_cnt│share_ct│
│    

## 🧪 Quick Quiz

1. **Why is user_id usually a good partition key?**

2. **What's the problem with using timestamp as a partition key?**

3. **When would you use vertical vs horizontal partitioning?**

In [8]:
print("📝 Quiz Answers")
print("=" * 50)
print()
print("1. Why user_id is good:")
print("   - High cardinality (millions of users)")
print("   - Users spread writes naturally")
print("   - Matches common access patterns")
print("   - User's data often accessed together")
print()
print("2. Problem with timestamp:")
print("   - All 'current' writes go to same shard")
print("   - Creates sequential hot spot")
print("   - Only latest shard is ever busy")
print()
print("3. Vertical vs Horizontal:")
print("   Vertical: Different access patterns per column")
print("            (content vs counters)")
print("   Horizontal: Same schema, too much data")
print("              (split rows across shards)")

📝 Quiz Answers

1. Why user_id is good:
   - High cardinality (millions of users)
   - Users spread writes naturally
   - Matches common access patterns
   - User's data often accessed together

2. Problem with timestamp:
   - All 'current' writes go to same shard
   - Creates sequential hot spot
   - Only latest shard is ever busy

3. Vertical vs Horizontal:
   Vertical: Different access patterns per column
            (content vs counters)
   Horizontal: Same schema, too much data
              (split rows across shards)


## 📚 Summary

### Key Takeaways

1. **Horizontal sharding** - Split rows across servers
2. **Choose keys wisely** - High cardinality, uniform distribution
3. **Hash for uniformity** - Spreads skewed keys evenly
4. **Vertical partitioning** - Separate by access pattern
5. **Avoid hot spots** - They defeat the purpose of sharding

### Next Up

In **Notebook 4**, we'll learn about queues and load shedding:
- Handling bursty traffic
- Async write patterns
- Graceful degradation